In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into memory (variables kept for later steps)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Convert timestamp columns to timezone-aware datetimes in UTC+8 for summaries
metric_df['_ts_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['_ts_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_df['_ts_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
error_logs_df['_ts_dt'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# --- metric.csv summary ---
metric_total_rows = metric_df.shape[0]
metric_min_ts = metric_df['_ts_dt'].min()
metric_max_ts = metric_df['_ts_dt'].max()
metric_unique_cmdb_sample = list(pd.Index(metric_df['cmdb_id'].unique())[:50])
metric_unique_kpi_sample = list(pd.Index(metric_df['kpi_name'].unique())[:50])
metric_distinct_pairs = metric_df[['cmdb_id', 'kpi_name']].drop_duplicates().shape[0]

metric_summary = pd.DataFrame({
    'total_rows': [metric_total_rows],
    'min_timestamp_UTC+8': [metric_min_ts],
    'max_timestamp_UTC+8': [metric_max_ts],
    'unique_cmdb_id_sample_up_to_50': [metric_unique_cmdb_sample],
    'unique_kpi_name_sample_up_to_50': [metric_unique_kpi_sample],
    'distinct_cmdb_kpi_pairs': [metric_distinct_pairs]
})

# --- trace.csv summary ---
trace_total_rows = trace_df.shape[0]
trace_min_ts = trace_df['_ts_dt'].min()
trace_max_ts = trace_df['_ts_dt'].max()
trace_unique_cmdb_sample = list(pd.Index(trace_df['cmdb_id'].unique())[:50])
trace_unique_trace_sample = list(pd.Index(trace_df['trace_name'].unique())[:50])
trace_distinct_pairs = trace_df[['cmdb_id', 'trace_name']].drop_duplicates().shape[0]

trace_summary = pd.DataFrame({
    'total_rows': [trace_total_rows],
    'min_timestamp_UTC+8': [trace_min_ts],
    'max_timestamp_UTC+8': [trace_max_ts],
    'unique_cmdb_id_sample_up_to_50': [trace_unique_cmdb_sample],
    'unique_trace_name_sample_up_to_50': [trace_unique_trace_sample],
    'distinct_cmdb_trace_pairs': [trace_distinct_pairs]
})

# --- log.csv summary ---
log_total_rows = log_df.shape[0]
log_min_ts = log_df['_ts_dt'].min()
log_max_ts = log_df['_ts_dt'].max()
log_unique_cmdb_sample = list(pd.Index(log_df['cmdb_id'].unique())[:50])
log_unique_log_sample = list(pd.Index(log_df['log_name'].unique())[:50])
log_distinct_pairs = log_df[['cmdb_id', 'log_name']].drop_duplicates().shape[0]

log_summary = pd.DataFrame({
    'total_rows': [log_total_rows],
    'min_timestamp_UTC+8': [log_min_ts],
    'max_timestamp_UTC+8': [log_max_ts],
    'unique_cmdb_id_sample_up_to_50': [log_unique_cmdb_sample],
    'unique_log_name_sample_up_to_50': [log_unique_log_sample],
    'distinct_cmdb_log_pairs': [log_distinct_pairs]
})

# --- error_logs.csv summary ---
error_total_rows = error_logs_df.shape[0]
error_min_ts = error_logs_df['_ts_dt'].min()
error_max_ts = error_logs_df['_ts_dt'].max()

# top up to 5 cmdb_id by count (compact)
error_cmdb_counts = (error_logs_df.groupby('cmdb_id')
                     .size()
                     .reset_index(name='count')
                     .sort_values('count', ascending=False)
                     .head(5))

# up to 5 sample messages: timestamp, cmdb_id, first 200 chars of message sorted by timestamp ascending
def first_200(s):
    return (s.fillna('').astype(str).str.slice(0, 200))

error_samples = (error_logs_df
                 .sort_values('_ts_dt', ascending=True)
                 .loc[:, ['_ts_dt', 'cmdb_id', 'message']]
                 .head(5)
                 .assign(message_sample=lambda df: first_200(df['message']))
                 .loc[:, ['_ts_dt', 'cmdb_id', 'message_sample']])

error_summary = pd.DataFrame({
    'total_rows': [error_total_rows],
    'min_timestamp_UTC+8': [error_min_ts],
    'max_timestamp_UTC+8': [error_max_ts]
})

# Final compact outputs (variables displayed for the controller). Keep them concise.
metric_summary, trace_summary, log_summary, error_summary, error_cmdb_counts, error_samples

```
Out[1]:
```
```python
# Compose a compact plain-English summary string based on previously loaded summaries.
metric_rows = int(metric_summary.at[0, 'total_rows'])
metric_min = metric_summary.at[0, 'min_timestamp_UTC+8']
metric_max = metric_summary.at[0, 'max_timestamp_UTC+8']
metric_pairs = int(metric_summary.at[0, 'distinct_cmdb_kpi_pairs'])

trace_rows = int(trace_summary.at[0, 'total_rows'])
trace_min = trace_summary.at[0, 'min_timestamp_UTC+8']
trace_max = trace_summary.at[0, 'max_timestamp_UTC+8']
trace_pairs = int(trace_summary.at[0, 'distinct_cmdb_trace_pairs'])

log_rows = int(log_summary.at[0, 'total_rows'])
log_min = log_summary.at[0, 'min_timestamp_UTC+8']
log_max = log_summary.at[0, 'max_timestamp_UTC+8']
log_pairs = int(log_summary.at[0, 'distinct_cmdb_log_pairs'])

error_rows = int(error_summary.at[0, 'total_rows'])
error_min = error_summary.at[0, 'min_timestamp_UTC+8']
error_max = error_summary.at[0, 'max_timestamp_UTC+8']

def fmt_ts(ts):
    return str(ts) if pd.notna(ts) else "N/A"

summary = (
    f"metric.csv: {metric_rows} rows; time range {fmt_ts(metric_min)} to {fmt_ts(metric_max)}; "
    f"{metric_pairs} distinct (cmdb_id, kpi_name) pairs.\n"
    f"trace.csv: {trace_rows} rows; time range {fmt_ts(trace_min)} to {fmt_ts(trace_max)}; "
    f"{trace_pairs} distinct (cmdb_id, trace_name) pairs.\n"
    f"log.csv: {log_rows} rows; time range {fmt_ts(log_min)} to {fmt_ts(log_max)}; "
    f"{log_pairs} distinct (cmdb_id, log_name) pairs.\n"
    f"error_logs.csv: {error_rows} rows; time range {fmt_ts(error_min)} to {fmt_ts(error_max)}; "
    f"No error log samples available." if error_rows == 0 else
    f"error_logs.csv: {error_rows} rows; time range {fmt_ts(error_min)} to {fmt_ts(error_max)}; samples available."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows       min_timestamp_UTC+8       max_timestamp_UTC+8                     unique_cmdb_id_sample_up_to_50                    unique_kpi_name_sample_up_to_50  distinct_cmdb_kpi_pairs
0      325324 2022-03-21 11:30:00+08:00 2022-03-21 12:30:00+08:00  [adservice, adservice-0, adservice-1, adservic...  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...                     5346,    total_rows       min_timestamp_UTC+8       max_timestamp_UTC+8                     unique_cmdb_id_sample_up_to_50                  unique_trace_name_sample_up_to_50  distinct_cmdb_trace_pairs
0       34628 2022-03-21 12:00:00+08:00 2022-03-21 12:29:00+08:00  [adservice-0, adservice-1, adservice-2, adserv...  [trace.from_frontend-0.duration_mean, trace.fr...                       1244,    total_rows       min_timestamp_UTC+8       max_timestamp_UTC+8                     unique_cmdb_id_sample_up_to_50   unique_log_name_sample_up_to_50  distinct_cmdb_log_pairs
0        1804 2022-03-21 12:00:00+08:00 2022-03-21 12:30:00+08:00  [adservice-0, adservice-1, adservice-2, cartse...  [log.error_count, log.row_count]                       62,    total_rows min_timestamp_UTC+8 max_timestamp_UTC+8
0           0                 NaT                 NaT, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [_ts_dt, cmdb_id, message_sample]
Index: [])```
```

In [ ]:
# In[2]:

# Candidate root-cause components (nodes, services, and pods) from provided lists
candidates = {
    # nodes
    "node-1","node-2","node-3","node-4","node-5","node-6",
    # services
    "frontend","shippingservice","checkoutservice","currencyservice","adservice",
    "emailservice","cartservice","productcatalogservice","recommendationservice","paymentservice",
    # pods (frontend, shippingservice, etc.)
    "frontend-0","frontend-1","frontend-2","frontend2-0",
    "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
    "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
    "adservice-0","adservice-1","adservice-2","adservice2-0",
    "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
    "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
    "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
    "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
    "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
}

# Reuse metric_df loaded previously; filter rows matching candidate cmdb_id values
metric_candidates = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# 1) total number of matching cmdb_id values
matched_cmdb_ids = sorted(metric_candidates['cmdb_id'].dropna().unique())
total_matched = len(matched_cmdb_ids)

# 2) Up to 50 matched cmdb_id values (alphabetically)
matched_cmdb_df = (pd.DataFrame({'cmdb_id': matched_cmdb_ids})
                   .sort_values('cmdb_id')
                   .head(50)
                   .reset_index(drop=True))

# 3) For each matched cmdb_id returned, count distinct kpi_name (top 20 by kpi count)
kpi_counts = (metric_candidates
              .groupby('cmdb_id')['kpi_name']
              .nunique()
              .reset_index(name='distinct_kpi_count')
              .sort_values(['distinct_kpi_count','cmdb_id'], ascending=[False, True])
              .reset_index(drop=True)
              .head(20))

# 4) Total number of distinct (cmdb_id, kpi_name) pairs among the matched set
total_pairs = metric_candidates[['cmdb_id', 'kpi_name']].drop_duplicates().shape[0]

# Compact outputs (keep displayed results small)
total_matched, matched_cmdb_df, kpi_counts, total_pairs, metric_candidates.shape

```
Out[2]:
```
# Compose a compact plain-English summary string using previously computed variables.
matched_list = matched_cmdb_df['cmdb_id'].tolist()  # up to 50 sorted cmdb_ids
top20_kpi_pairs = [f"{row.cmdb_id}: {row.distinct_kpi_count}" for row in kpi_counts.itertuples(index=False)]
rows, cols = metric_candidates.shape

summary = (
    f"Total matching cmdb_id values: {total_matched}.\n\n"
    f"Matched cmdb_id values (up to 50, alphabetical):\n"
    f"{', '.join(matched_list)}\n\n"
    f"Top 20 matched cmdb_ids by distinct kpi_name count (cmdb_id: distinct_kpi_count):\n"
    f"{'; '.join(top20_kpi_pairs)}\n\n"
    f"Total distinct (cmdb_id, kpi_name) pairs among matched set: {total_pairs}.\n\n"
    f"Filtered metric rows for these matched cmdb_ids are stored in variable 'metric_candidates' "
    f"with shape: {rows} rows x {cols} columns."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(56,                      cmdb_id
0                  adservice
1                adservice-0
2                adservice-1
3                adservice-2
4               adservice2-0
..                       ...
45  productcatalogservice2-0
46     recommendationservice
47   recommendationservice-0
48   recommendationservice-1
49   recommendationservice-2

[50 rows x 1 columns],                      cmdb_id  distinct_kpi_count
0                  adservice                 341
1                frontend2-0                 204
2                 frontend-0                 192
3                 frontend-1                 192
4                 frontend-2                 192
5          checkoutservice-1                 148
6         checkoutservice2-0                 148
7          checkoutservice-0                 144
8          checkoutservice-2                 144
9    productcatalogservice-0                 120
10   productcatalogservice-1                 120
11   productcatalogservice-2                 116
12  productcatalogservice2-0                 104
13              adservice2-0                 102
14  recommendationservice2-0                 102
15             cartservice-0                 100
16             cartservice-2                 100
17            cartservice2-0                 100
18   recommendationservice-0                  98
19   recommendationservice-1                  98, 4989, (303547, 5))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse metric_candidates already in kernel
# Compute aggregated global statistics per (cmdb_id, kpi_name)
group = metric_candidates.groupby(['cmdb_id', 'kpi_name'])['value']

component_kpi_thresholds = (
    group.agg(
        count='count',
        mean='mean',
        std='std',
        min='min',
        max='max',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# Compute severity = abs(P95 - mean) / (abs(mean) + 1e-9)
component_kpi_thresholds['severity'] = (
    (component_kpi_thresholds['p95'] - component_kpi_thresholds['mean']).abs()
    / (component_kpi_thresholds['mean'].abs() + 1e-9)
)

# Prepare compact top-20 table sorted by severity desc, select required columns
top20 = (
    component_kpi_thresholds
    .sort_values('severity', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'count', 'mean', 'std', 'min', 'max', 'p95', 'p5', 'severity']]
    .head(20)
    .reset_index(drop=True)
)

# For compact display, round numeric columns reasonably
numeric_cols = ['mean', 'std', 'min', 'max', 'p95', 'p5', 'severity']
top20[numeric_cols] = top20[numeric_cols].round(6)

# Output: store full results in 'component_kpi_thresholds' (already stored) and display top20
component_kpi_thresholds, top20

```
Out[3]:
```
# Compose a compact plain-English summary of the results
total_pairs = int(component_kpi_thresholds.shape[0])

# Recreate top20 sorted by severity to ensure consistency
top20_df = (component_kpi_thresholds
            .sort_values('severity', ascending=False)
            .loc[:, ['cmdb_id', 'kpi_name', 'count', 'mean', 'p95', 'p5', 'severity']]
            .head(20)
            .reset_index(drop=True))

# Helper to truncate long kpi names for compactness
def trim(s, n=80):
    s = str(s)
    return s if len(s) <= n else s[:n-3] + "..."

lines = [
    f"Computed global stats for each (cmdb_id, kpi_name) pair and stored full table in variable 'component_kpi_thresholds'.",
    f"Total distinct (cmdb_id, kpi_name) pairs analyzed: {total_pairs}.",
    "",
    "Top 20 pairs by severity (severity = |P95 - mean| / (|mean|+1e-9)) — columns: cmdb_id | kpi_name | count | mean | p95 | p5 | severity:",
]

for i, row in top20_df.iterrows():
    lines.append(
        f"{i+1:2d}. {row['cmdb_id']} | {trim(row['kpi_name'], 80)} | {int(row['count'])} | "
        f"{row['mean']:.6g} | {row['p95']:.6g} | {row['p5']:.6g} | {row['severity']:.6g}"
    )

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  count          mean           std           min           max           p95            p5  severity
0              adservice                                     app.grpc.count     61    214.081967     53.298310     58.000000    255.000000    252.000000     65.000000  0.177119
1              adservice                                       app.grpc.mrt     61      0.002741      0.000080      0.002613      0.003006      0.002912      0.002631  0.062176
2              adservice                                        app.grpc.rr     61    100.000000      0.000000    100.000000    100.000000    100.000000    100.000000  0.000000
3              adservice                                        app.grpc.sr     61    100.000000      0.000000    100.000000    100.000000    100.000000    100.000000  0.000000
4              adservice                                     app.http.count     61     24.000000      0.000000     24.000000     24.000000     24.000000     24.000000  0.000000
...                  ...                                                ...    ...           ...           ...           ...           ...           ...           ...       ...
4984  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     61     24.909836      7.148140      4.500000     36.000000     34.000000      8.000000  0.364923
4985  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61  67314.344262  18179.146970  13725.000000  91825.000000  90400.000000  26975.000000  0.342953
4986  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61      0.000964      0.001476      0.000000      0.005775      0.004525      0.000000  3.694299
4987  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61     20.549180      5.133797      4.000000     27.500000     27.000000      9.500000  0.313921
4988  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61   2043.102459    518.002553    447.750000   2786.000000   2736.250000    945.250000  0.339262

[4989 rows x 10 columns],               cmdb_id                                           kpi_name  count        mean          std  min          max          p95        p5   severity
0              node-4                                 system.io.avg_q_sz     61    0.000656     0.002496  0.0     0.010000     0.010000  0.000000  14.249978
1      cartservice2-0    container.node-5.container_fs_reads_MB./dev/vda     61    0.015945     0.056788  0.0     0.259766     0.214844  0.000000  12.473895
2       cartservice-2  container.node-6.container_cpu_cfs_throttled_s...     61    0.001401     0.005385  0.0     0.024310     0.018406  0.000000  12.142380
3          frontend-0  container.node-6.container_memory_failures.hie...     61    2.704918    10.936703  0.0    66.000000    33.000000  0.000000  11.200000
4          frontend-0  container.node-6.container_memory_failures.con...     61    2.704918    10.936703  0.0    66.000000    33.000000  0.000000  11.200000
5        adservice2-0       container.node-6.container_fs_reads./dev/vda     61    0.040984     0.165708  0.0     1.000000     0.500000  0.000000  11.200000
6    paymentservice-0      container.node-6.container_cpu_system_seconds     61    0.000820     0.002611  0.0     0.010000     0.010000  0.000000  11.199986
7        adservice2-0    container.node-6.container_fs_reads_MB./dev/vda     61    0.000160     0.000647  0.0     0.003906     0.001953  0.000000  11.199930
8              node-4                                      system.io.w_s     61    0.836066     2.968896  0.0    13.000000    10.000000  0.000000  10.960784
9      cartservice2-0       container.node-5.container_fs_reads./dev/vda     61    0.803279     3.008708  0.0    16.000000     9.500000  0.000000  10.826531
10        frontend2-0   container.node-6.container_fs_writes_MB./dev/vda     61    0.003778     0.014870  0.0     0.072266     0.042969  0.000000  10.372879
11        frontend2-0  container.node-6.container_memory_failures.con...     61  464.028689  1594.835932  0.0  7804.500000  5263.500000  0.000000  10.343049
12        frontend2-0  container.node-6.container_memory_failures.hie...     61  464.028689  1594.835932  0.0  7804.500000  5263.500000  0.000000  10.343049
13         frontend-1    container.node-5.container_fs_reads_MB./dev/vda     61    1.059234     4.172841  0.0    20.466797    11.849609  0.000000  10.186960
14        frontend2-0  container.node-6.container_cpu_cfs_throttled_s...     61    0.261023     0.817609  0.0     3.701450     2.892181  0.002528  10.080185
15         frontend-2    container.node-5.container_fs_reads_MB./dev/vda     61    0.933594     3.726487  0.0    18.630859     9.820312  0.000000   9.518828
16         frontend-1  container.node-5.container_memory_failures.hie...     61    1.622951     6.534304  0.0    33.000000    16.500000  0.000000   9.166667
17         frontend-1  container.node-5.container_memory_failures.con...     61    1.622951     6.534304  0.0    33.000000    16.500000  0.000000   9.166667
18         frontend-2      container.node-5.container_fs_writes./dev/vda     61    0.196721     0.792037  0.0     4.000000     2.000000  0.000000   9.166667
19  currencyservice-0  container.node-6.container_cpu_cfs_throttled_p...     61    0.049180     0.175711  0.0     1.000000     0.500000  0.000000   9.166666)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) Filter metric_candidates to incident window and store in metric_incident
start = pd.Timestamp('2022-03-21 12:00:00').tz_localize(tz)
end = pd.Timestamp('2022-03-21 12:30:00').tz_localize(tz)

# metric_candidates already exists in kernel and contains '_ts_dt' timezone-aware column
metric_incident = metric_candidates[
    (metric_candidates['_ts_dt'] >= start) & (metric_candidates['_ts_dt'] <= end)
].copy()

# 2) Join metric_incident with component_kpi_thresholds to attach mean, p95, p5
# Use only the required threshold columns to avoid extra duplicates
threshold_cols = ['cmdb_id', 'kpi_name', 'mean', 'p95', 'p5']
metric_anomalies = metric_incident.merge(
    component_kpi_thresholds[threshold_cols],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# 3) Create anomaly flags per row
metric_anomalies['anomaly_high'] = metric_anomalies['value'] >= metric_anomalies['p95']
metric_anomalies['anomaly_low'] = metric_anomalies['value'] <= metric_anomalies['p5']

# Ensure boolean (no NA treated as False)
metric_anomalies['anomaly_high'] = metric_anomalies['anomaly_high'].fillna(False)
metric_anomalies['anomaly_low'] = metric_anomalies['anomaly_low'].fillna(False)

# 4) For each (cmdb_id, kpi_name) compute aggregates and longest consecutive anomaly streak
def compute_group_summary(df):
    # df is rows for a single (cmdb_id,kpi_name)
    mask = df['anomaly_high'] | df['anomaly_low']
    total_anoms = int(mask.sum())
    if total_anoms == 0:
        earliest = pd.NaT
    else:
        earliest = df.loc[mask, '_ts_dt'].min()
    # anomaly type
    saw_high = bool(df['anomaly_high'].any())
    saw_low = bool(df['anomaly_low'].any())
    if saw_high and saw_low:
        a_type = 'both'
    elif saw_high:
        a_type = 'high'
    elif saw_low:
        a_type = 'low'
    else:
        a_type = 'none'
    # longest consecutive anomaly streak in minutes
    times = df.loc[mask, '_ts_dt']
    if times.empty:
        longest = 0
    else:
        # convert to integer seconds since epoch
        secs = (times.astype('int64') // 10**9).sort_values().values
        if len(secs) == 0:
            longest = 0
        else:
            if len(secs) == 1:
                longest = 1
            else:
                diffs = np.diff(secs)
                # segment breaks where diff != 60
                breaks = np.where(diffs != 60)[0]
                starts = np.concatenate(([0], breaks + 1))
                ends = np.concatenate((breaks, [len(secs)-1]))
                lengths = ends - starts + 1
                longest = int(lengths.max())
    # include global mean,p95,p5 (they are the same for all rows in this df after join)
    mean = df['mean'].iloc[0] if 'mean' in df.columns else np.nan
    p95 = df['p95'].iloc[0] if 'p95' in df.columns else np.nan
    p5 = df['p5'].iloc[0] if 'p5' in df.columns else np.nan

    return pd.Series({
        'total_anomalies': total_anoms,
        'earliest_anomaly_timestamp': earliest,
        'anomaly_type': a_type,
        'longest_consecutive_anomaly_streak_minutes': longest,
        'mean': mean,
        'p95': p95,
        'p5': p5
    })

component_kpi_anomaly_summary = (
    metric_anomalies
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .apply(lambda g: compute_group_summary(g))
    .reset_index()
)

# Ensure earliest timestamp column is timezone-aware Asia/Shanghai (it should be)
# Format earliest timestamp column name as requested
component_kpi_anomaly_summary = component_kpi_anomaly_summary.rename(
    columns={'earliest_anomaly_timestamp': 'earliest_anomaly_timestamp_UTC+8'}
)

# 5) Prepare top 20 rows sorted by total_anomalies descending and select required columns
top20 = (
    component_kpi_anomaly_summary
    .sort_values('total_anomalies', ascending=False)
    .loc[:, [
        'cmdb_id',
        'kpi_name',
        'total_anomalies',
        'earliest_anomaly_timestamp_UTC+8',
        'anomaly_type',
        'longest_consecutive_anomaly_streak_minutes',
        'mean',
        'p95',
        'p5'
    ]]
    .head(20)
    .reset_index(drop=True)
)

# Round numeric fields for compactness
for col in ['mean', 'p95', 'p5']:
    if col in top20.columns:
        top20[col] = top20[col].round(6)

# Final displayed result: top20 DataFrame
metric_incident.shape, metric_anomalies.shape, component_kpi_anomaly_summary.shape, top20

```
Out[4]:
```
Summary of the incident-window anomaly analysis (plain English):

- I filtered metric data into the incident window 2022-03-21 12:00:00+08 through 2022-03-21 12:30:00+08 and stored it in variable metric_incident (shape: 154,289 rows × 5 columns).
- I joined those rows with the global thresholds and created per-row anomaly flags, storing the joined table in metric_anomalies (shape: 154,289 rows × 10 columns).
- I aggregated anomalies per (cmdb_id, kpi_name) and stored the full per-pair summary in component_kpi_anomaly_summary (4989 distinct pairs).

Top findings (top 20 pairs by total_anomalies):
- Many of the top pairs (examples: shippingservice2-0 mesh.* series and multiple adservice KPIs) show total_anomalies = 31 — i.e., anomalies present at every minute of the 31-minute window.
- Earliest anomaly for these top pairs = 2022-03-21 12:00:00+08 (the start of the window).
- Anomaly type for these top pairs is mostly "both" (both high and low flags occur), indicating values equal to p95 and/or p5 during the window.
- Longest consecutive anomaly streak for the top pairs = 31 minutes (the full window), indicating continuous anomalies across the incident period.
- The summary rows include mean, p95, and p5 (values shown in the stored summary); many of the top entries have p95 == p5 == the same constant, which explains repeated equal-value anomalies across the window.

Stored variables for follow-up:
- metric_incident (filtered rows for the incident window)
- metric_anomalies (joined rows with mean/p95/p5 and per-row anomaly flags)
- component_kpi_anomaly_summary (aggregated per (cmdb_id, kpi_name) anomaly summary)

If you want, I can:
- show the top 20 summary table as a DataFrame, or
- list which services/pods have the most distinct KPIs with anomalies (service-level aggregation).

The original code execution output of IPython Kernel is also provided below for reference:

((154289, 5), (154289, 10), (4989, 10),                cmdb_id                                           kpi_name  total_anomalies earliest_anomaly_timestamp_UTC+8 anomaly_type  longest_consecutive_anomaly_streak_minutes          mean        p95         p5
0   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...               31        2022-03-21 12:00:00+08:00         both                                          31  0.000000e+00        0.0        0.0
1   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...               31        2022-03-21 12:00:00+08:00         both                                          31  0.000000e+00        0.0        0.0
2   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...               31        2022-03-21 12:00:00+08:00         both                                          31  0.000000e+00        0.0        0.0
3            adservice                                        app.grpc.rr               31        2022-03-21 12:00:00+08:00         both                                          31  1.000000e+02      100.0      100.0
4            adservice                                        app.grpc.sr               31        2022-03-21 12:00:00+08:00         both                                          31  1.000000e+02      100.0      100.0
5            adservice                                     app.http.count               31        2022-03-21 12:00:00+08:00         both                                          31  2.400000e+01       24.0       24.0
6   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...               31        2022-03-21 12:00:00+08:00         both                                          31  0.000000e+00        0.0        0.0
7            adservice                                        app.http.rr               31        2022-03-21 12:00:00+08:00         both                                          31  1.000000e+02      100.0      100.0
8            adservice                                        app.http.sr               31        2022-03-21 12:00:00+08:00         both                                          31  1.000000e+02      100.0      100.0
9            adservice    runtime.java_lang_ClassLoading_LoadedClassCount               31        2022-03-21 12:00:00+08:00         both                                          31  5.162000e+03     5162.0     5162.0
10           adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...               31        2022-03-21 12:00:00+08:00         both                                          31  5.185000e+03     5185.0     5185.0
11           adservice  runtime.java_lang_ClassLoading_UnloadedClassCount               31        2022-03-21 12:00:00+08:00         both                                          31  2.300000e+01       23.0       23.0
12           adservice             runtime.java_lang_ClassLoading_Verbose               31        2022-03-21 12:00:00+08:00         both                                          31  0.000000e+00        0.0        0.0
13           adservice  runtime.java_lang_Compilation_CompilationTimeM...               31        2022-03-21 12:00:00+08:00         both                                          31  1.000000e+00        1.0        1.0
14           adservice  runtime.java_lang_Compilation_TotalCompilation...               31        2022-03-21 12:00:00+08:00         both                                          31  4.702100e+04    47021.0    47021.0
15  shippingservice2-0  container.node-6.container_memory_failures.con...               31        2022-03-21 12:00:00+08:00         both                                          31  4.868852e+00       33.0        0.0
16  shippingservice2-0  container.node-6.container_memory_failures.con...               31        2022-03-21 12:00:00+08:00         both                                          31  0.000000e+00        0.0        0.0
17  shippingservice2-0  container.node-6.container_memory_failures.hie...               31        2022-03-21 12:00:00+08:00         both                                          31  4.868852e+00       33.0        0.0
18  shippingservice2-0  container.node-6.container_memory_failures.hie...               31        2022-03-21 12:00:00+08:00         both                                          31  0.000000e+00        0.0        0.0
19  shippingservice2-0      container.node-6.container_memory_mapped_file               31        2022-03-21 12:00:00+08:00         both                                          31  1.216512e+06  1216512.0  1216512.0)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variable: component_kpi_anomaly_summary
# Ensure the DataFrame exists
_comp = component_kpi_anomaly_summary

# Aggregation per cmdb_id
component_anomaly_summary = (
    _comp
    .groupby('cmdb_id', as_index=False)
    .agg(
        distinct_anomalous_kpis = ('kpi_name', lambda s: int(( _comp.loc[s.index, 'total_anomalies'] > 0 ).sum()) ),
        total_anomaly_points = ('total_anomalies', 'sum'),
        earliest_anomaly_timestamp_UTC_plus8 = ('earliest_anomaly_timestamp_UTC+8', 'min'),
        max_consecutive_streak_minutes = ('longest_consecutive_anomaly_streak_minutes', 'max'),
        full_window_kpi_count = ('longest_consecutive_anomaly_streak_minutes', lambda x: int((x == 31).sum()))
    )
)

# The lambda for distinct_anomalous_kpis above used a safe approach; 
# but to ensure correctness (count distinct kpi_name where total_anomalies>0), recompute clearly:
_component = _comp.copy()
_component['has_anom'] = _component['total_anomalies'] > 0
distinct_counts = (_component[_component['has_anom']]
                   .groupby('cmdb_id')['kpi_name']
                   .nunique()
                   .reset_index(name='distinct_anomalous_kpis'))

# Merge to ensure accurate distinct_anomalous_kpis
component_anomaly_summary = component_anomaly_summary.drop(columns=['distinct_anomalous_kpis']).merge(
    distinct_counts, on='cmdb_id', how='left'
)
component_anomaly_summary['distinct_anomalous_kpis'] = component_anomaly_summary['distinct_anomalous_kpis'].fillna(0).astype(int)

# Order columns as requested and ensure timestamp timezone is Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
# If timestamps are not tz-aware, attempt to localize (safe-check)
def ensure_tz(ts):
    if pd.isna(ts):
        return pd.NaT
    if hasattr(ts, 'tzinfo') and ts.tzinfo is not None:
        return ts.tz_convert(tz)
    else:
        return pd.to_datetime(ts).tz_localize(tz)

component_anomaly_summary['earliest_anomaly_timestamp_UTC+8'] = component_anomaly_summary['earliest_anomaly_timestamp_UTC_plus8'].apply(ensure_tz)
component_anomaly_summary = component_anomaly_summary.drop(columns=['earliest_anomaly_timestamp_UTC_plus8'])

# Sort and select top 20 as requested
component_anomaly_summary_top20 = (
    component_anomaly_summary
    .sort_values(['distinct_anomalous_kpis', 'total_anomaly_points'], ascending=[False, False])
    .loc[:, [
        'cmdb_id',
        'distinct_anomalous_kpis',
        'total_anomaly_points',
        'earliest_anomaly_timestamp_UTC+8',
        'max_consecutive_streak_minutes',
        'full_window_kpi_count'
    ]]
    .head(20)
    .reset_index(drop=True)
)

# Keep output compact: display only the top20 DataFrame
component_anomaly_summary, component_anomaly_summary_top20

```
Out[5]:
```
```python
# Compose a concise plain-English summary from component_anomaly_summary_top20
rows = component_anomaly_summary_top20.copy()

total_components = component_anomaly_summary.shape[0]

lines = [
    f"Aggregated per-component anomaly summary (total components analyzed: {total_components}).",
    "Top components by number of distinct anomalous KPIs (showing up to 20):"
]

for i, r in rows.iterrows():
    ts = r['earliest_anomaly_timestamp_UTC+8']
    ts_str = ts.strftime('%Y-03-%d %H:%M:%S%z') if pd.notna(ts) else "N/A"
    lines.append(
        f"{i+1:2d}. {r['cmdb_id']}: {int(r['distinct_anomalous_kpis'])} distinct anomalous KPIs, "
        f"{int(r['total_anomaly_points'])} total anomaly points, earliest anomaly {ts_str}, "
        f"max consecutive streak {int(r['max_consecutive_streak_minutes'])} min, "
        f"{int(r['full_window_kpi_count'])} KPIs anomalous for full 31-min window."
    )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id  total_anomaly_points  max_consecutive_streak_minutes  full_window_kpi_count  distinct_anomalous_kpis earliest_anomaly_timestamp_UTC+8
0            adservice                  9564                              31                    297                      341        2022-03-21 12:00:00+08:00
1          adservice-0                  2310                              31                     66                       94        2022-03-21 12:00:00+08:00
2          adservice-1                  2073                              31                     59                       86        2022-03-21 12:00:00+08:00
3          adservice-2                  2317                              31                     67                       94        2022-03-21 12:00:00+08:00
4         adservice2-0                  2460                              31                     72                      102        2022-03-21 12:00:00+08:00
..                 ...                   ...                             ...                    ...                      ...                              ...
51     shippingservice                    79                              31                      2                        4        2022-03-21 12:00:00+08:00
52   shippingservice-0                  2190                              31                     62                       92        2022-03-21 12:00:00+08:00
53   shippingservice-1                  1917                              31                     55                       84        2022-03-21 12:00:00+08:00
54   shippingservice-2                  1941                              31                     55                       84        2022-03-21 12:00:00+08:00
55  shippingservice2-0                  1960                              31                     55                       84        2022-03-21 12:00:00+08:00

[56 rows x 6 columns],                      cmdb_id  distinct_anomalous_kpis  total_anomaly_points earliest_anomaly_timestamp_UTC+8  max_consecutive_streak_minutes  full_window_kpi_count
0                  adservice                      341                  9564        2022-03-21 12:00:00+08:00                              31                    297
1                frontend2-0                      198                  4512        2022-03-21 12:00:00+08:00                              31                    125
2                 frontend-1                      189                  4237        2022-03-21 12:00:00+08:00                              31                    119
3                 frontend-2                      189                  4112        2022-03-21 12:00:00+08:00                              31                    118
4                 frontend-0                      188                  4095        2022-03-21 12:00:00+08:00                              31                    118
5          checkoutservice-1                      148                  3185        2022-03-21 12:00:00+08:00                              31                     90
6         checkoutservice2-0                      148                  3156        2022-03-21 12:00:00+08:00                              31                     90
7          checkoutservice-0                      144                  3097        2022-03-21 12:00:00+08:00                              31                     86
8          checkoutservice-2                      141                  2916        2022-03-21 12:00:00+08:00                              31                     83
9    productcatalogservice-1                      120                  2749        2022-03-21 12:00:00+08:00                              31                     77
10   productcatalogservice-0                      120                  2729        2022-03-21 12:00:00+08:00                              31                     77
11   productcatalogservice-2                      116                  2586        2022-03-21 12:00:00+08:00                              31                     73
12  productcatalogservice2-0                      104                  2442        2022-03-21 12:00:00+08:00                              31                     68
13  recommendationservice2-0                      102                  2586        2022-03-21 12:00:00+08:00                              31                     76
14              adservice2-0                      102                  2460        2022-03-21 12:00:00+08:00                              31                     72
15             cartservice-2                      100                  2358        2022-03-21 12:00:00+08:00                              31                     70
16             cartservice-0                      100                  2348        2022-03-21 12:00:00+08:00                              31                     70
17            cartservice2-0                      100                  2303        2022-03-21 12:00:00+08:00                              31                     62
18   recommendationservice-0                       98                  2451        2022-03-21 12:00:00+08:00                              31                     72
19   recommendationservice-1                       98                  2446        2022-03-21 12:00:00+08:00                              31                     72)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global statistics for trace.csv per (cmdb_id, trace_name)
trace_group = trace_df.groupby(['cmdb_id', 'trace_name'])['value']

trace_thresholds = (
    trace_group.agg(
        count='count',
        mean='mean',
        std='std',
        min='min',
        max='max',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# 2) Filter trace.csv to the incident window and store in trace_incident (reuse variable name pattern)
start = pd.Timestamp('2022-03-21 12:00:00').tz_localize(tz)
end = pd.Timestamp('2022-03-21 12:30:00').tz_localize(tz)

trace_incident = trace_df[(trace_df['_ts_dt'] >= start) & (trace_df['_ts_dt'] <= end)].copy()

# 3) Join trace_incident with trace_thresholds to attach p95 and mean
trace_anomalies = trace_incident.merge(
    trace_thresholds[['cmdb_id', 'trace_name', 'mean', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# 4) Flag per-row anomaly: trace_anomaly_high = value >= p95
trace_anomalies['trace_anomaly_high'] = (trace_anomalies['value'] >= trace_anomalies['p95']).fillna(False)

# 5) Aggregate trace_anomalies per cmdb_id
# total_trace_anomaly_points: count of trace_anomaly_high True
# distinct_trace_names_with_anomaly: number of distinct trace_name with any True
# earliest_trace_anomaly_timestamp_UTC+8: min timestamp where anomaly occurs

# Compute per-row marker for grouping distinct trace names
trace_anom_mask = trace_anomalies['trace_anomaly_high']

per_cmdb = (
    trace_anomalies[trace_anom_mask]
    .groupby('cmdb_id')
    .agg(
        total_trace_anomaly_points = ('trace_anomaly_high', 'sum'),
        distinct_trace_names_with_anomaly = ('trace_name', 'nunique'),
        earliest_trace_anomaly_timestamp_UTC_plus8 = ('_ts_dt', 'min')
    )
    .reset_index()
)

# Ensure timezone-awareness for earliest timestamps (should already be tz-aware)
def ensure_tz(ts):
    if pd.isna(ts):
        return pd.NaT
    if getattr(ts, 'tzinfo', None) is None:
        return pd.to_datetime(ts).tz_localize(tz)
    return ts.tz_convert(tz)

per_cmdb['earliest_trace_anomaly_timestamp_UTC+8'] = per_cmdb['earliest_trace_anomaly_timestamp_UTC_plus8'].apply(ensure_tz)
per_cmdb = per_cmdb.drop(columns=['earliest_trace_anomaly_timestamp_UTC_plus8'])

# 6) Prepare top 20 cmdb_id rows sorted by total_trace_anomaly_points desc
top20_trace_components = (
    per_cmdb
    .sort_values('total_trace_anomaly_points', ascending=False)
    .loc[:, [
        'cmdb_id',
        'total_trace_anomaly_points',
        'distinct_trace_names_with_anomaly',
        'earliest_trace_anomaly_timestamp_UTC+8'
    ]]
    .head(20)
    .reset_index(drop=True)
)

# Store final variables in kernel as requested:
# trace_thresholds, trace_anomalies, trace_incident, top20_trace_components

# For compact display, return shapes for large tables and the top20 result
trace_thresholds.shape, trace_anomalies.shape, trace_incident.shape, top20_trace_components

```
Out[6]:
```
# Compose a concise plain-English summary of the trace anomaly results using existing kernel variables.
# Reuse: trace_thresholds, trace_incident, trace_anomalies, top20_trace_components

n_pairs = trace_thresholds.shape[0]
incident_rows = trace_incident.shape[0]
anomaly_rows = trace_anomalies.shape[0]

lines = [
    f"Trace global statistics computed for {n_pairs} distinct (cmdb_id, trace_name) pairs.",
    f"Incident window rows: {incident_rows}; joined trace rows with thresholds: {anomaly_rows}.",
    "",
    "Top trace-anomalous components (by total_trace_anomaly_points):"
]

for i, r in top20_trace_components.iterrows():
    ts = r['earliest_trace_anomaly_timestamp_UTC+8']
    ts_str = ts.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(ts) else "N/A"
    lines.append(
        f"{i+1:2d}. {r['cmdb_id']}: {int(r['total_trace_anomaly_points'])} anomaly points, "
        f"{int(r['distinct_trace_names_with_anomaly'])} distinct trace names, earliest {ts_str}"
    )

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

((1244, 9), (34628, 8), (34628, 5),                     cmdb_id  total_trace_anomaly_points  distinct_trace_names_with_anomaly earliest_trace_anomaly_timestamp_UTC+8
0         checkoutservice-0                         958                                 92              2022-03-21 12:00:00+08:00
1         checkoutservice-2                         949                                 92              2022-03-21 12:00:00+08:00
2                frontend-1                         895                                 92              2022-03-21 12:00:00+08:00
3         checkoutservice-1                         883                                 92              2022-03-21 12:00:00+08:00
4                frontend-0                         868                                 92              2022-03-21 12:00:00+08:00
5                frontend-2                         863                                 92              2022-03-21 12:00:00+08:00
6             cartservice-1                         474                                 28              2022-03-21 12:00:00+08:00
7             cartservice-0                         472                                 28              2022-03-21 12:00:00+08:00
8   productcatalogservice-0                         389                                 44              2022-03-21 12:00:00+08:00
9   productcatalogservice-1                         381                                 44              2022-03-21 12:00:00+08:00
10  productcatalogservice-2                         373                                 44              2022-03-21 12:00:00+08:00
11            cartservice-2                         361                                 28              2022-03-21 12:00:00+08:00
12       checkoutservice2-0                         358                                 36              2022-03-21 12:00:00+08:00
13              frontend2-0                         357                                 36              2022-03-21 12:00:00+08:00
14  recommendationservice-2                         268                                 28              2022-03-21 12:00:00+08:00
15  recommendationservice-0                         260                                 28              2022-03-21 12:00:00+08:00
16  recommendationservice-1                         259                                 28              2022-03-21 12:00:00+08:00
17        shippingservice-0                         232                                 24              2022-03-21 12:00:00+08:00
18        shippingservice-1                         230                                 24              2022-03-21 12:00:00+08:00
19        currencyservice-1                         223                                 24              2022-03-21 12:00:00+08:00)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously computed DataFrames:
# - component_anomaly_summary (per-component metric aggregates)
# - per_cmdb (per-component trace aggregates)

# Ensure expected columns exist and normalize column names for clarity
_metric_df = component_anomaly_summary.copy()
if 'earliest_anomaly_timestamp_UTC+8' in _metric_df.columns:
    _metric_df = _metric_df.rename(columns={'earliest_anomaly_timestamp_UTC+8': 'earliest_metric_anomaly_timestamp_UTC+8'})
elif 'earliest_anomaly_timestamp_UTC_plus8' in _metric_df.columns:
    _metric_df = _metric_df.rename(columns={'earliest_anomaly_timestamp_UTC_plus8': 'earliest_metric_anomaly_timestamp_UTC+8'})

_trace_df = per_cmdb.copy()
# per_cmdb already has 'earliest_trace_anomaly_timestamp_UTC+8' from previous step

# Outer join to include components present in either source
component_multi_modal_summary = pd.merge(
    _metric_df[[
        'cmdb_id',
        'distinct_anomalous_kpis',
        'total_anomaly_points',
        'earliest_metric_anomaly_timestamp_UTC+8'
    ]],
    _trace_df[[
        'cmdb_id',
        'total_trace_anomaly_points',
        'distinct_trace_names_with_anomaly',
        'earliest_trace_anomaly_timestamp_UTC+8'
    ]],
    on='cmdb_id',
    how='outer'
)

# Fill missing numeric fields with 0 and ensure integer types where appropriate
component_multi_modal_summary['distinct_anomalous_kpis'] = component_multi_modal_summary['distinct_anomalous_kpis'].fillna(0).astype(int)
component_multi_modal_summary['total_anomaly_points'] = component_multi_modal_summary['total_anomaly_points'].fillna(0).astype(int)
component_multi_modal_summary['total_trace_anomaly_points'] = component_multi_modal_summary['total_trace_anomaly_points'].fillna(0).astype(int)
component_multi_modal_summary['distinct_trace_names_with_anomaly'] = component_multi_modal_summary['distinct_trace_names_with_anomaly'].fillna(0).astype(int)

# Ensure earliest timestamp columns are timezone-aware Asia/Shanghai or NaT
def ensure_tz_or_nat(ts):
    if pd.isna(ts):
        return pd.NaT
    try:
        ts = pd.to_datetime(ts)
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        return ts.tz_convert(tz)
    except Exception:
        return pd.NaT

component_multi_modal_summary['earliest_metric_anomaly_timestamp_UTC+8'] = component_multi_modal_summary['earliest_metric_anomaly_timestamp_UTC+8'].apply(ensure_tz_or_nat)
component_multi_modal_summary['earliest_trace_anomaly_timestamp_UTC+8'] = component_multi_modal_summary['earliest_trace_anomaly_timestamp_UTC+8'].apply(ensure_tz_or_nat)

# Prepare top 20 rows sorted by distinct_anomalous_kpis desc, then total_trace_anomaly_points desc
component_multi_modal_summary_top20 = (
    component_multi_modal_summary
    .sort_values(['distinct_anomalous_kpis', 'total_trace_anomaly_points'], ascending=[False, False])
    .loc[:, [
        'cmdb_id',
        'distinct_anomalous_kpis',
        'total_anomaly_points',
        'earliest_metric_anomaly_timestamp_UTC+8',
        'total_trace_anomaly_points',
        'distinct_trace_names_with_anomaly',
        'earliest_trace_anomaly_timestamp_UTC+8'
    ]]
    .head(20)
    .reset_index(drop=True)
)

# Display the compact top-20 result
component_multi_modal_summary_top20

```
Out[7]:
```
```python
# Compose a concise plain-English summary string from the previously computed top-20 multi-modal table
rows = component_multi_modal_summary_top20.copy()

lines = [
    f"Cross-modal component summary (showing top {len(rows)} components):",
    ""
]

for i, r in rows.iterrows():
    em_ts = r['earliest_metric_anomaly_timestamp_UTC+8']
    et_ts = r['earliest_trace_anomaly_timestamp_UTC+8']
    em_str = em_ts.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(em_ts) else "N/A"
    et_str = et_ts.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(et_ts) else "N/A"
    lines.append(
        f"{i+1:2d}. {r['cmdb_id']}: "
        f"{r['distinct_anomalous_kpis']} distinct metric KPIs anomalous, "
        f"{r['total_anomaly_points']} metric anomaly points; earliest metric anomaly {em_str}; "
        f"{r['total_trace_anomaly_points']} trace anomaly points across {r['distinct_trace_names_with_anomaly']} trace names; "
        f"earliest trace anomaly {et_str}."
    )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  distinct_anomalous_kpis  total_anomaly_points earliest_metric_anomaly_timestamp_UTC+8  total_trace_anomaly_points  distinct_trace_names_with_anomaly earliest_trace_anomaly_timestamp_UTC+8
0                  adservice                      341                  9564               2022-03-21 12:00:00+08:00                           0                                  0                                    NaT
1                frontend2-0                      198                  4512               2022-03-21 12:00:00+08:00                         357                                 36              2022-03-21 12:00:00+08:00
2                 frontend-1                      189                  4237               2022-03-21 12:00:00+08:00                         895                                 92              2022-03-21 12:00:00+08:00
3                 frontend-2                      189                  4112               2022-03-21 12:00:00+08:00                         863                                 92              2022-03-21 12:00:00+08:00
4                 frontend-0                      188                  4095               2022-03-21 12:00:00+08:00                         868                                 92              2022-03-21 12:00:00+08:00
5          checkoutservice-1                      148                  3185               2022-03-21 12:00:00+08:00                         883                                 92              2022-03-21 12:00:00+08:00
6         checkoutservice2-0                      148                  3156               2022-03-21 12:00:00+08:00                         358                                 36              2022-03-21 12:00:00+08:00
7          checkoutservice-0                      144                  3097               2022-03-21 12:00:00+08:00                         958                                 92              2022-03-21 12:00:00+08:00
8          checkoutservice-2                      141                  2916               2022-03-21 12:00:00+08:00                         949                                 92              2022-03-21 12:00:00+08:00
9    productcatalogservice-0                      120                  2729               2022-03-21 12:00:00+08:00                         389                                 44              2022-03-21 12:00:00+08:00
10   productcatalogservice-1                      120                  2749               2022-03-21 12:00:00+08:00                         381                                 44              2022-03-21 12:00:00+08:00
11   productcatalogservice-2                      116                  2586               2022-03-21 12:00:00+08:00                         373                                 44              2022-03-21 12:00:00+08:00
12  productcatalogservice2-0                      104                  2442               2022-03-21 12:00:00+08:00                          72                                  8              2022-03-21 12:00:00+08:00
13  recommendationservice2-0                      102                  2586               2022-03-21 12:00:00+08:00                         183                                 20              2022-03-21 12:00:00+08:00
14              adservice2-0                      102                  2460               2022-03-21 12:00:00+08:00                          36                                  4              2022-03-21 12:00:00+08:00
15             cartservice-0                      100                  2348               2022-03-21 12:00:00+08:00                         472                                 28              2022-03-21 12:00:00+08:00
16             cartservice-2                      100                  2358               2022-03-21 12:00:00+08:00                         361                                 28              2022-03-21 12:00:00+08:00
17            cartservice2-0                      100                  2303               2022-03-21 12:00:00+08:00                         188                                 12              2022-03-21 12:00:00+08:00
18   recommendationservice-2                       98                  2440               2022-03-21 12:00:00+08:00                         268                                 28              2022-03-21 12:00:00+08:00
19   recommendationservice-0                       98                  2451               2022-03-21 12:00:00+08:00                         260                                 28              2022-03-21 12:00:00+08:00```
```

In [ ]:
# In[8]:

# Identify components with anomalies in BOTH metrics and traces using existing summary tables
# Reuse variables from kernel: component_multi_modal_summary, trace_incident, trace_anomalies

# 1) Filter components where distinct_anomalous_kpis > 0 AND total_trace_anomaly_points > 0
both_mask = (
    (component_multi_modal_summary['distinct_anomalous_kpis'] > 0) &
    (component_multi_modal_summary['total_trace_anomaly_points'] > 0)
)
multi_modal_df = component_multi_modal_summary.loc[both_mask].copy()

# Total count
total_multi_modal_components = int(multi_modal_df.shape[0])

# 2) Prepare top 20 components sorted by distinct_anomalous_kpis desc, then total_trace_anomaly_points desc
top_components = (
    multi_modal_df
    .sort_values(['distinct_anomalous_kpis', 'total_trace_anomaly_points'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'distinct_anomalous_kpis', 'total_trace_anomaly_points']]
    .head(20)
    .reset_index(drop=True)
)

# 3) For each returned cmdb_id, attach up to 20 distinct trace_name values observed in the incident-window trace data
# Use trace_incident (incident-window trace rows) if available; fallback to trace_anomalies if needed
trace_src = trace_incident if 'trace_incident' in globals() else trace_anomalies

def sample_trace_names(cmdb_id, src_df, limit=20):
    names = src_df.loc[src_df['cmdb_id'] == cmdb_id, 'trace_name'].dropna().unique()
    return list(names[:limit])

top_components['sample_trace_names'] = top_components['cmdb_id'].apply(lambda cid: sample_trace_names(cid, trace_src, 20))

# Store the list of identified multi-modal components for later use
multi_modal_candidates = multi_modal_df[['cmdb_id']].drop_duplicates().reset_index(drop=True)

# Compact outputs: total count, top components with sample trace names, and multi_modal_candidates (list)
total_multi_modal_components, top_components, multi_modal_candidates.head(20)

```
Out[8]:
```
```python
# Compose a plain-English summary of multi-modal components (metrics AND traces)
# Reuse existing kernel variables: total_multi_modal_components, top_components

def trim(s, n=60):
    s = str(s)
    return s if len(s) <= n else s[:n-3] + "..."

# Build summary lines
lines = []
count = int(total_multi_modal_components) if 'total_multi_modal_components' in globals() else (len(multi_modal_candidates) if 'multi_modal_candidates' in globals() else 0)
lines.append(f"Total components with anomalies in BOTH metrics and traces: {count}.\n")

lines.append("Up to 20 components (sorted by metric anomalous KPI count):")
if 'top_components' in globals() and not top_components.empty:
    for i, row in top_components.reset_index(drop=True).iterrows():
        cid = row['cmdb_id']
        mk = int(row['distinct_anomalous_kpis'])
        tp = int(row['total_trace_anomaly_points'])
        trace_list = row.get('sample_trace_names', [])
        # show up to 20 trace names, each trimmed for compactness
        trace_sample = ", ".join(trim(n, 60) for n in (trace_list[:20] if isinstance(trace_list, (list, tuple)) else []))
        lines.append(f"{i+1:2d}. {cid}: {mk} distinct anomalous metric KPIs; {tp} trace anomaly points; sample traces: [{trace_sample}]")
else:
    lines.append("No multi-modal components found or 'top_components' unavailable.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(40,                      cmdb_id  distinct_anomalous_kpis  total_trace_anomaly_points                                 sample_trace_names
0                frontend2-0                      198                         357  [trace.from_root.duration_mean, trace.from_roo...
1                 frontend-1                      189                         895  [trace.from_root.duration_mean, trace.from_roo...
2                 frontend-2                      189                         863  [trace.from_root.duration_mean, trace.from_roo...
3                 frontend-0                      188                         868  [trace.from_root.duration_mean, trace.from_roo...
4          checkoutservice-1                      148                         883  [trace.from_frontend-0.duration_mean, trace.fr...
5         checkoutservice2-0                      148                         358  [trace.from_frontend2-0.duration_mean, trace.f...
6          checkoutservice-0                      144                         958  [trace.from_frontend-0.duration_mean, trace.fr...
7          checkoutservice-2                      141                         949  [trace.from_frontend-0.duration_mean, trace.fr...
8    productcatalogservice-0                      120                         389  [trace.from_checkoutservice-0.duration_mean, t...
9    productcatalogservice-1                      120                         381  [trace.from_checkoutservice-0.duration_mean, t...
10   productcatalogservice-2                      116                         373  [trace.from_checkoutservice-0.duration_mean, t...
11  productcatalogservice2-0                      104                          72  [trace.from_checkoutservice2-0.duration_mean, ...
12  recommendationservice2-0                      102                         183  [trace.from_frontend2-0.duration_mean, trace.f...
13              adservice2-0                      102                          36  [trace.from_frontend2-0.duration_mean, trace.f...
14             cartservice-0                      100                         472  [trace.from_checkoutservice-0.duration_mean, t...
15             cartservice-2                      100                         361  [trace.from_checkoutservice-0.duration_mean, t...
16            cartservice2-0                      100                         188  [trace.from_checkoutservice2-0.duration_mean, ...
17   recommendationservice-2                       98                         268  [trace.from_frontend-0.duration_mean, trace.fr...
18   recommendationservice-0                       98                         260  [trace.from_frontend-0.duration_mean, trace.fr...
19   recommendationservice-1                       98                         259  [trace.from_frontend-0.duration_mean, trace.fr...,                cmdb_id
0          adservice-0
1          adservice-1
2          adservice-2
3         adservice2-0
4        cartservice-0
5        cartservice-1
6        cartservice-2
7       cartservice2-0
8    checkoutservice-0
9    checkoutservice-1
10   checkoutservice-2
11  checkoutservice2-0
12   currencyservice-0
13   currencyservice-1
14   currencyservice-2
15  currencyservice2-0
16      emailservice-0
17      emailservice-1
18      emailservice-2
19     emailservice2-0)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: multi_modal_candidates (DataFrame with cmdb_id), trace_incident (incident-window trace rows)
candidate_ids = set(multi_modal_candidates['cmdb_id'].dropna().astype(str).unique())

# 1) Filter trace_incident to rows whose cmdb_id is in multi_modal_candidates
trace_incident_filtered = trace_incident[trace_incident['cmdb_id'].isin(candidate_ids)].copy()

# 2) Parse trace_name for pattern 'trace.from_<source>.' and extract '<source>' into column 'trace_from'
# Pattern: starts with 'trace.from_' then capture any characters up to the next dot.
trace_incident_filtered['trace_from'] = trace_incident_filtered['trace_name'].astype(str).str.extract(r'^trace\.from_([^\.]+)\.', expand=False)

# 3) Aggregate per cmdb_id
def sample_list(sr, limit=10):
    vals = pd.Index(sr.dropna().unique())
    return list(vals[:limit])

agg = (
    trace_incident_filtered
    .groupby('cmdb_id', as_index=False)
    .agg(
        distinct_from_count = ('trace_from', lambda s: int(s.dropna().nunique())),
        sample_from_list = ('trace_from', lambda s: sample_list(s, 10)),
        total_trace_rows = ('trace_name', 'count'),
        earliest_trace_row_timestamp = ('_ts_dt', 'min')
    )
)

# Ensure timezone-awareness for earliest timestamp and rename column
def ensure_tz_or_nat(ts):
    if pd.isna(ts):
        return pd.NaT
    if getattr(ts, 'tzinfo', None) is None:
        return pd.to_datetime(ts).tz_localize(tz)
    return ts.tz_convert(tz)

agg['earliest_trace_row_timestamp_UTC+8'] = agg['earliest_trace_row_timestamp'].apply(ensure_tz_or_nat)
agg = agg.drop(columns=['earliest_trace_row_timestamp'])

# 4) Store results in requested kernel variables
trace_incident_parsed = trace_incident_filtered  # full per-row parsed table (keeps trace_from)
trace_call_relations = agg.sort_values(['distinct_from_count', 'total_trace_rows'], ascending=[False, False]).reset_index(drop=True)

# Compact table to return: top 20 rows
trace_call_relations_top20 = trace_call_relations.loc[:, [
    'cmdb_id',
    'distinct_from_count',
    'sample_from_list',
    'total_trace_rows',
    'earliest_trace_row_timestamp_UTC+8'
]].head(20)

# Output compact summary: top20 aggregated rows and shape of parsed table
trace_call_relations_top20, trace_incident_parsed.shape

```
Out[9]:
```
```python
# Compose a concise plain-English summary of the trace call relation analysis
top = trace_call_relations_top20 if 'trace_call_relations_top20' in globals() else trace_call_relations.head(20)
parsed_shape = trace_incident_parsed.shape if 'trace_incident_parsed' in globals() else ("N/A","N/A")
num_multi = int(multi_modal_candidates.shape[0]) if 'multi_modal_candidates' in globals() else 0

lines = [
    f"Parsed trace call relations for {num_multi} multi-modal components; parsed incident trace rows filtered: {parsed_shape[0]} rows.",
    "",
    "Top findings (up to 20 components):"
]

# show a few concise examples from top table (up to 8)
for i, r in top.head(8).iterrows():
    sample = r['sample_from_list'] if isinstance(r['sample_from_list'], (list, tuple)) else []
    sample_str = ", ".join(sample[:6])  # show up to 6 callers for compactness
    lines.append(
        f"{i+1}. {r['cmdb_id']}: {int(r['distinct_from_count'])} distinct caller sources; "
        f"{int(r['total_trace_rows'])} trace rows; sample callers: [{sample_str}]"
    )

lines.append("")
lines.append("Overall observation: most affected components receive calls from checkoutservice and frontend pods (common callers),")
lines.append("and several components (e.g., productcatalogservice-0/1/2) show the largest diversity of callers (11 distinct sources).")
lines.append("Earliest trace rows for these components begin at the incident window start: 2022-03-21 12:00:00+08.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  distinct_from_count                                   sample_from_list  total_trace_rows earliest_trace_row_timestamp_UTC+8
0   productcatalogservice-1                   11  [checkoutservice-0, checkoutservice-1, checkou...              1176          2022-03-21 12:00:00+08:00
1   productcatalogservice-0                   11  [checkoutservice-0, checkoutservice-1, checkou...              1172          2022-03-21 12:00:00+08:00
2   productcatalogservice-2                   11  [checkoutservice-0, checkoutservice-1, checkou...              1172          2022-03-21 12:00:00+08:00
3             cartservice-1                    6  [checkoutservice-0, checkoutservice-1, checkou...               808          2022-03-21 12:00:00+08:00
4             cartservice-0                    6  [checkoutservice-0, checkoutservice-1, checkou...               804          2022-03-21 12:00:00+08:00
5             cartservice-2                    6  [checkoutservice-0, checkoutservice-1, checkou...               796          2022-03-21 12:00:00+08:00
6         currencyservice-1                    6  [checkoutservice-0, checkoutservice-1, checkou...               696          2022-03-21 12:00:00+08:00
7         currencyservice-0                    6  [checkoutservice-0, checkoutservice-1, checkou...               692          2022-03-21 12:00:00+08:00
8         currencyservice-2                    6  [checkoutservice-0, checkoutservice-1, checkou...               692          2022-03-21 12:00:00+08:00
9         shippingservice-1                    6  [checkoutservice-0, checkoutservice-1, checkou...               684          2022-03-21 12:00:00+08:00
10        shippingservice-2                    6  [checkoutservice-0, checkoutservice-1, checkou...               680          2022-03-21 12:00:00+08:00
11        shippingservice-0                    6  [checkoutservice-0, checkoutservice-1, checkou...               672          2022-03-21 12:00:00+08:00
12        checkoutservice-2                    4         [frontend-0, frontend-1, frontend-2, root]              2444          2022-03-21 12:00:00+08:00
13        checkoutservice-0                    4         [frontend-0, frontend-1, frontend-2, root]              2372          2022-03-21 12:00:00+08:00
14        checkoutservice-1                    4         [frontend-0, frontend-2, root, frontend-1]              2288          2022-03-21 12:00:00+08:00
15  recommendationservice-0                    3               [frontend-0, frontend-1, frontend-2]               840          2022-03-21 12:00:00+08:00
16  recommendationservice-1                    3               [frontend-0, frontend-1, frontend-2]               840          2022-03-21 12:00:00+08:00
17  recommendationservice-2                    3               [frontend-0, frontend-1, frontend-2]               840          2022-03-21 12:00:00+08:00
18              adservice-0                    3               [frontend-0, frontend-1, frontend-2]               360          2022-03-21 12:00:00+08:00
19              adservice-1                    3               [frontend-0, frontend-1, frontend-2]               360          2022-03-21 12:00:00+08:00, (34628, 6))```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Target components (include service-level name if present)
targets = [
    "productcatalogservice-0",
    "productcatalogservice-1",
    "productcatalogservice-2",
    "productcatalogservice",
    "productcatalogservice2-0"
]

# Reuse existing component_kpi_anomaly_summary from kernel, filter rows for targets
productcatalog_kpi_anomalies = component_kpi_anomaly_summary[
    component_kpi_anomaly_summary['cmdb_id'].isin(targets)
].copy()

# Ensure earliest timestamp column exists and is timezone-aware UTC+8
ts_col = 'earliest_anomaly_timestamp_UTC+8'
if ts_col in productcatalog_kpi_anomalies.columns:
    def ensure_tz_or_nat(ts):
        if pd.isna(ts):
            return pd.NaT
        ts = pd.to_datetime(ts)
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        return ts.tz_convert(tz)
    productcatalog_kpi_anomalies[ts_col] = productcatalog_kpi_anomalies[ts_col].apply(ensure_tz_or_nat)
else:
    productcatalog_kpi_anomalies[ts_col] = pd.NaT

# Select and format columns for compact output
cols = [
    'cmdb_id',
    'kpi_name',
    'total_anomalies',
    ts_col,
    'anomaly_type',
    'longest_consecutive_anomaly_streak_minutes',
    'mean',
    'p95',
    'p5'
]

# Prepare top 20 rows sorted by total_anomalies desc
top20 = (
    productcatalog_kpi_anomalies
    .loc[:, cols]
    .sort_values('total_anomalies', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Round numeric columns for compact display
for c in ['mean', 'p95', 'p5']:
    if c in top20.columns:
        top20[c] = top20[c].round(6)

# Display only the top20 as the result; full filtered set is stored in productcatalog_kpi_anomalies
top20

```
Out[10]:
```
```python
# Compose a concise plain-English summary of the productcatalog KPIs anomaly results
pc_df = productcatalog_kpi_anomalies  # from kernel
total_rows = int(pc_df.shape[0])
distinct_cmdb = sorted(pc_df['cmdb_id'].dropna().unique())
num_cmdb = len(distinct_cmdb)

# Count how many KPI pairs had anomalies every minute (longest_consecutive_anomaly_streak_minutes == 31)
full_window_count = int((pc_df['longest_consecutive_anomaly_streak_minutes'] == 31).sum())

# Earliest anomaly timestamp among these rows
earliest_ts = pc_df['earliest_anomaly_timestamp_UTC+8'].min()
earliest_ts_str = str(earliest_ts) if pd.notna(earliest_ts) else "N/A"

# Top affected cmdb_ids by total_anomalies (up to 5)
top_cmdb = (
    pc_df.groupby('cmdb_id', as_index=False)['total_anomalies']
    .sum()
    .sort_values('total_anomalies', ascending=False)
)
top_cmdb_list = top_cmdb['cmdb_id'].tolist()[:5]

# Prepare short examples of KPI types frequently seen
example_kpis = pc_df['kpi_name'].dropna().astype(str).value_counts().head(6).index.tolist()

summary = (
    f"productcatalog KPIs anomaly summary:\n"
    f"- Total KPI pairs examined: {total_rows} across {num_cmdb} component entries ({', '.join(distinct_cmdb)}).\n"
    f"- {full_window_count} KPI pairs were anomalous for the full 31-minute incident window (longest streak = 31 min).\n"
    f"- Earliest anomaly observed: {earliest_ts_str} (UTC+8).\n"
    f"- Top affected components (examples): {', '.join(top_cmdb_list)}.\n"
    f"- Representative KPI types in the top results: {', '.join(example_kpis)}.\n"
    f"Note: Many top entries show anomaly_type='both' and identical mean/p95/p5 values, indicating repeated constant-valued signals throughout the window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_anomalies earliest_anomaly_timestamp_UTC+8 anomaly_type  longest_consecutive_anomaly_streak_minutes       mean        p95         p5
0   productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
1   productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
2   productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
3      productcatalogservice                                        app.grpc.rr               31        2022-03-21 12:00:00+08:00         both                                          31      100.0      100.0      100.0
4      productcatalogservice                                        app.grpc.sr               31        2022-03-21 12:00:00+08:00         both                                          31      100.0      100.0      100.0
5   productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
6    productcatalogservice-0  container.node-6.container_cpu_cfs_throttled_p...               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
7    productcatalogservice-0  container.node-6.container_cpu_cfs_throttled_s...               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
8    productcatalogservice-0    container.node-6.container_cpu_load_average_10s               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
9   productcatalogservice2-0     container.node-6.container_tasks_state.running               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
10  productcatalogservice2-0    container.node-6.container_tasks_state.sleeping               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
11  productcatalogservice2-0     container.node-6.container_tasks_state.stopped               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
12  productcatalogservice2-0  container.node-6.container_tasks_state.uninter...               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
13  productcatalogservice2-0                 container.node-6.container_threads               31        2022-03-21 12:00:00+08:00         both                                          31       36.0       36.0       36.0
14  productcatalogservice2-0             container.node-6.container_threads_max               31        2022-03-21 12:00:00+08:00         both                                          31        0.0        0.0        0.0
15  productcatalogservice2-0  container.node-6.container_ulimits_soft.max_op...               31        2022-03-21 12:00:00+08:00         both                                          31  1048576.0  1048576.0  1048576.0
16  productcatalogservice2-0         container.node-6.container_spec_cpu_period               31        2022-03-21 12:00:00+08:00         both                                          31   100000.0   100000.0   100000.0
17  productcatalogservice2-0          container.node-6.container_spec_cpu_quota               31        2022-03-21 12:00:00+08:00         both                                          31    20000.0    20000.0    20000.0
18  productcatalogservice2-0         container.node-6.container_spec_cpu_shares               31        2022-03-21 12:00:00+08:00         both                                          31      102.0      102.0      102.0
19  productcatalogservice2-0    container.node-6.container_spec_memory_limit_MB               31        2022-03-21 12:00:00+08:00         both                                          31      128.0      128.0      128.0```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse productcatalog_kpi_anomalies from kernel
df = productcatalog_kpi_anomalies.copy()

# Ensure earliest timestamp column exists and is timezone-aware UTC+8
ts_col = 'earliest_anomaly_timestamp_UTC+8'
if ts_col not in df.columns:
    df[ts_col] = pd.NaT

def ensure_tz_or_nat(ts):
    if pd.isna(ts):
        return pd.NaT
    ts = pd.to_datetime(ts)
    if ts.tzinfo is None:
        return ts.tz_localize(tz)
    return ts.tz_convert(tz)

df[ts_col] = df[ts_col].apply(ensure_tz_or_nat)

# Categorization function using substring matches (case-insensitive)
def categorize_kpi(name):
    if pd.isna(name):
        return 'other'
    s = str(name).lower()
    # cpu category
    if ('cfs_throttled' in s) or ('cpu' in s) or ('cpu_load' in s) or ('cpu_system' in s) or ('cpu_user' in s):
        return 'cpu'
    # memory category
    if ('memory' in s) or ('memory_failures' in s) or ('memory_limit' in s) or ('rss' in s) or ('mapped_file' in s) or ('mappedfile' in s):
        return 'memory'
    # fs_io category
    if ('fs_reads' in s) or ('fs_writes' in s) or ('fs_reads_mb' in s) or ('fs_writes_mb' in s) or ('io' in s) or ('w_s' in s) or ('r_s' in s) or ('reads' in s) or ('writes' in s):
        return 'fs_io'
    # threads/tasks/ulimits
    if ('threads' in s) or ('tasks' in s) or ('ulimits' in s) or ('tasks_state' in s) or ('threads_max' in s):
        return 'threads_tasks_ulimits'
    # jvm/runtime
    if ('java' in s) or ('runtime' in s) or ('jvm' in s) or ('classloading' in s) or ('compilation' in s) or ('heapmemory' in s) or ('heap_memory' in s):
        return 'jvm_runtime'
    # mesh/network
    if ('mesh.' in s) or ('mesh.source' in s) or ('mesh.destination' in s) or ('jaeger' in s) or ('trace' in s):
        return 'mesh_network'
    # fallback
    return 'other'

# Apply categorization
df['category'] = df['kpi_name'].apply(categorize_kpi)

# Store full categorized rows in kernel variable
productcatalog_kpi_categorized = df

# Aggregation per category
def sample_kpis(series, limit=10):
    vals = list(pd.Index(series.dropna().unique()))
    return vals[:limit]

agg = (
    productcatalog_kpi_categorized
    .groupby('category', as_index=False)
    .agg(
        affected_cmdb_count = ('cmdb_id', lambda s: int(s.dropna().nunique())),
        kpi_pair_count = ('kpi_name', lambda s: int(s.dropna().nunique())),
        full_window_kpi_count = ('longest_consecutive_anomaly_streak_minutes', lambda s: int((s == 31).sum())),
        earliest_anomaly_timestamp_UTC_plus8 = (ts_col, lambda s: s.min()),
        sample_kpis = ('kpi_name', lambda s: sample_kpis(s, 10))
    )
)

# Rename timestamp column to requested name
agg = agg.rename(columns={'earliest_anomaly_timestamp_UTC_plus8': 'earliest_anomaly_timestamp_UTC+8'})

# Ensure timestamp column is timezone-aware and sorted
agg['earliest_anomaly_timestamp_UTC+8'] = agg['earliest_anomaly_timestamp_UTC+8'].apply(ensure_tz_or_nat)

# Order columns as requested
category_summary = agg.loc[:, [
    'category',
    'affected_cmdb_count',
    'kpi_pair_count',
    'full_window_kpi_count',
    'earliest_anomaly_timestamp_UTC+8',
    'sample_kpis'
]]

# Sort for presentation (by kpi_pair_count desc)
category_summary = category_summary.sort_values(['kpi_pair_count', 'full_window_kpi_count'], ascending=[False, False]).reset_index(drop=True)

# Keep output compact: show the category summary (<=20 rows)
productcatalog_kpi_categorized.shape, category_summary

```
Out[11]:
```
```python
# Compose a concise plain-English summary string of the categorization results
total_rows = int(productcatalog_kpi_categorized.shape[0])
category_rows = category_summary.copy()

lines = [
    f"productcatalog KPI categorization: {total_rows} KPI-pairs classified across categories.",
    ""
]

for _, r in category_rows.iterrows():
    cat = r['category']
    ac = int(r['affected_cmdb_count'])
    kp = int(r['kpi_pair_count'])
    fw = int(r['full_window_kpi_count'])
    ts = r['earliest_anomaly_timestamp_UTC+8']
    ts_str = ts.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(ts) else "N/A"
    samples = r['sample_kpis'] if isinstance(r['sample_kpis'], (list, tuple)) else []
    sample_str = ", ".join(str(s) for s in samples[:6])
    lines.append(f"- {cat}: {kp} KPI pairs across {ac} components; {fw} were anomalous for full 31-min window; earliest {ts_str}; sample KPIs: [{sample_str}]")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((464, 11),                 category  affected_cmdb_count  kpi_pair_count  full_window_kpi_count earliest_anomaly_timestamp_UTC+8                                        sample_kpis
0                  fs_io                    4             114                    156        2022-03-21 12:00:00+08:00  [container.node-6.container_fs_io_current./dev...
1                  other                    5              20                     42        2022-03-21 12:00:00+08:00  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...
2                 memory                    4              15                     57        2022-03-21 12:00:00+08:00  [container.node-6.container_memory_cache, cont...
3                    cpu                    4              10                     22        2022-03-21 12:00:00+08:00  [container.node-6.container_cpu_cfs_periods, c...
4  threads_tasks_ulimits                    4               5                     20        2022-03-21 12:00:00+08:00  [container.node-6.container_tasks_state.runnin...)```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Target cmdb_ids and incident window
targets = [
    "productcatalogservice",
    "productcatalogservice-0",
    "productcatalogservice-1",
    "productcatalogservice-2",
    "productcatalogservice2-0"
]
start = pd.Timestamp('2022-03-21 12:00:00').tz_localize(tz)
end = pd.Timestamp('2022-03-21 12:30:00').tz_localize(tz)

# Ensure log_df has timezone-aware timestamp column '_ts_dt'
if '_ts_dt' not in globals().get('log_df', pd.DataFrame()).columns:
    log_df['_ts_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # If exists but not tz-aware, convert/ensure
    try:
        if log_df['_ts_dt'].dt.tz is None:
            log_df['_ts_dt'] = pd.to_datetime(log_df['_ts_dt']).dt.tz_localize(tz)
        else:
            log_df['_ts_dt'] = log_df['_ts_dt'].dt.tz_convert(tz)
    except Exception:
        log_df['_ts_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Filter logs to incident window and target components; store in productcatalog_logs
productcatalog_logs = log_df[
    (log_df['cmdb_id'].isin(targets)) &
    (log_df['_ts_dt'] >= start) &
    (log_df['_ts_dt'] <= end)
].copy()

# 2) Aggregate by log_name: count, earliest_timestamp_UTC+8, mean(value)
log_agg = (
    productcatalog_logs
    .groupby('log_name', as_index=False)
    .agg(
        count = ('log_name', 'count'),
        earliest_timestamp_UTC_plus8 = ('_ts_dt', 'min'),
        mean_value = ('value', 'mean')
    )
)

# Rename timestamp column to requested name and round mean for compactness
log_agg = log_agg.rename(columns={'earliest_timestamp_UTC_plus8': 'earliest_timestamp_UTC+8'})
log_agg['mean_value'] = log_agg['mean_value'].round(6)

# Top 20 by count descending
top_logs = log_agg.sort_values('count', ascending=False).loc[:, ['log_name','count','earliest_timestamp_UTC+8','mean_value']].head(20).reset_index(drop=True)

# 3) Up to 5 sample rows from productcatalog_logs showing timestamp_UTC+8, cmdb_id, log_name, value sorted by timestamp ascending
if not productcatalog_logs.empty:
    sample_logs = (
        productcatalog_logs
        .sort_values('_ts_dt', ascending=True)
        .loc[:, ['_ts_dt','cmdb_id','log_name','value']]
        .head(5)
        .reset_index(drop=True)
    )
    # rename timestamp column for clarity
    sample_logs = sample_logs.rename(columns={'_ts_dt': 'timestamp_UTC+8'})
else:
    sample_logs = productcatalog_logs.head(0).loc[:, ['timestamp','cmdb_id','log_name','value']]

# Return compact outputs (and productcatalog_logs stored in kernel)
top_logs, sample_logs

```
Out[12]:
```
```python
# Compose a concise plain-English summary of the filtered productcatalog logs
top = top_logs.copy()
samples = sample_logs.copy()

lines = [
    "Filtered logs for productcatalogservice components during 2022-03-21 12:00:00+08 to 12:30:00+08:",
    "",
    "1) Top log aggregate (by count):"
]

for _, r in top.iterrows():
    ts = r['earliest_timestamp_UTC+8']
    ts_str = ts.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(ts) else "N/A"
    lines.append(f"- {r['log_name']}: count={int(r['count'])}, earliest={ts_str}, mean_value={r['mean_value']}")

lines += [
    "",
    "2) Sample log rows (earliest 5 rows):"
]

for _, r in samples.iterrows():
    ts = r['timestamp_UTC+8']
    ts_str = ts.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(ts) else "N/A"
    lines.append(f"- {ts_str} | {r['cmdb_id']} | {r['log_name']} | value={r['value']}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          log_name  count  earliest_timestamp_UTC+8  mean_value
0  log.error_count     90 2022-03-21 12:00:00+08:00         0.0
1    log.row_count     90 2022-03-21 12:00:00+08:00       782.6,             timestamp_UTC+8                  cmdb_id         log_name  value
0 2022-03-21 12:00:00+08:00  productcatalogservice-0  log.error_count    0.0
1 2022-03-21 12:00:00+08:00  productcatalogservice-0    log.row_count  970.0
2 2022-03-21 12:00:00+08:00  productcatalogservice-1  log.error_count    0.0
3 2022-03-21 12:00:00+08:00  productcatalogservice-1    log.row_count  971.0
4 2022-03-21 12:00:00+08:00  productcatalogservice-2  log.error_count    0.0)```
```

In [ ]:
# In[13]:

import re
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse productcatalog_kpi_anomalies from kernel
_df = productcatalog_kpi_anomalies.copy()

# Ensure earliest timestamp column exists and is timezone-aware
ts_col = 'earliest_anomaly_timestamp_UTC+8'
if ts_col not in _df.columns:
    _df[ts_col] = pd.NaT

def ensure_tz_or_nat(ts):
    if pd.isna(ts):
        return pd.NaT
    ts = pd.to_datetime(ts)
    if ts.tzinfo is None:
        return ts.tz_localize(tz)
    return ts.tz_convert(tz)

_df[ts_col] = _df[ts_col].apply(ensure_tz_or_nat)

# Extract node tokens like 'node-<number>' from kpi_name (case-insensitive)
_df['node_tokens'] = _df['kpi_name'].astype(str).str.findall(r'(node-\d+)', flags=re.IGNORECASE)

# Explode to have one row per (original row, node_token) for grouping; drop rows with no tokens
exploded = _df.explode('node_tokens')
exploded = exploded[exploded['node_tokens'].notna() & (exploded['node_tokens'] != '')].copy()

# If no node references found, create empty mapping
if exploded.empty:
    productcatalog_node_map = pd.DataFrame(columns=[
        'node_token',
        'count_of_kpi_pairs_referencing_node',
        'earliest_anomaly_timestamp_UTC+8',
        'example_kpi_names'
    ])
else:
    # Group by node token and compute aggregations
    grouped = (
        exploded
        .groupby('node_tokens', as_index=False)
        .agg(
            count_of_kpi_pairs_referencing_node = ('kpi_name', 'size'),  # count of (cmdb_id,kpi_name) rows referencing node
            earliest_anomaly_timestamp_UTC_plus8 = (ts_col, 'min'),
            example_kpi_names = ('kpi_name', lambda s: list(pd.Index(s.dropna().unique())[:10]))
        )
    )

    # Rename timestamp column and ensure tz
    grouped = grouped.rename(columns={'node_tokens': 'node_token', 'earliest_anomaly_timestamp_UTC_plus8': 'earliest_anomaly_timestamp_UTC+8'})
    grouped['earliest_anomaly_timestamp_UTC+8'] = grouped['earliest_anomaly_timestamp_UTC+8'].apply(ensure_tz_or_nat)

    # Sort by count desc and store full mapping
    productcatalog_node_map = grouped.sort_values('count_of_kpi_pairs_referencing_node', ascending=False).reset_index(drop=True)

# Store top 10 for compact return
productcatalog_node_map_top10 = productcatalog_node_map.head(10).loc[:, [
    'node_token',
    'count_of_kpi_pairs_referencing_node',
    'earliest_anomaly_timestamp_UTC+8',
    'example_kpi_names'
]]

# Display compact result (variable output)
productcatalog_node_map, productcatalog_node_map_top10

```
Out[13]:
```
```python
# Summarize node-token findings for productcatalog KPIs (plain English) and display as a single string.
top = productcatalog_node_map_top10 if 'productcatalog_node_map_top10' in globals() else productcatalog_node_map

if top.empty:
    summary = "No node tokens (like 'node-<number>') were found in productcatalog KPI names."
else:
    row = top.iloc[0]
    node = row['node_token']
    count = int(row['count_of_kpi_pairs_referencing_node'])
    ts = row['earliest_anomaly_timestamp_UTC+8']
    ts_str = ts.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(ts) else "N/A"
    examples = row.get('example_kpi_names', []) or []
    sample_examples = ", ".join(str(x) for x in examples[:5])
    summary = (
        f"Node-token analysis for productcatalog KPIs:\n"
        f"- Identified node: {node}\n"
        f"- Number of KPI pairs referencing this node: {count}\n"
        f"- Earliest anomaly timestamp among those KPI pairs: {ts_str} (UTC+8)\n"
        f"- Example KPI names referencing the node (up to 5): {sample_examples}\n\n"
        f"Conclusion: productcatalogservice pods are associated with host {node}; next check node-level I/O and disk metrics on {node}."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  node_token  count_of_kpi_pairs_referencing_node earliest_anomaly_timestamp_UTC+8                                  example_kpi_names
0     node-6                                  256        2022-03-21 12:00:00+08:00  [container.node-6.container_cpu_cfs_periods, c...,   node_token  count_of_kpi_pairs_referencing_node earliest_anomaly_timestamp_UTC+8                                  example_kpi_names
0     node-6                                  256        2022-03-21 12:00:00+08:00  [container.node-6.container_cpu_cfs_periods, c...)```
```

In [ ]:
# In[14]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) Select thresholds for node-6 from component_kpi_thresholds
node6_thresholds = component_kpi_thresholds[component_kpi_thresholds['cmdb_id'] == 'node-6'][
    ['cmdb_id', 'kpi_name', 'mean', 'p95', 'p5']
].rename(columns={'mean': 'threshold_mean', 'p95': 'threshold_p95', 'p5': 'threshold_p5'})

# 2) Filter incident-window metric rows for node-6 and join thresholds
# Use metric_incident (already filtered to candidate cmdb_ids and incident window)
node6_rows = metric_incident[metric_incident['cmdb_id'] == 'node-6'].copy()

# Join to attach thresholds (some kpis may not have thresholds; keep them as NaN)
node6_rows = node6_rows.merge(node6_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Create per-row anomaly flags using threshold_p95 and threshold_p5
node6_rows['anomaly_high'] = (node6_rows['value'] >= node6_rows['threshold_p95']).fillna(False)
node6_rows['anomaly_low'] = (node6_rows['value'] <= node6_rows['threshold_p5']).fillna(False)
# combined anomaly mask
node6_rows['anomaly_any'] = node6_rows['anomaly_high'] | node6_rows['anomaly_low']

# 4) Aggregate per (cmdb_id, kpi_name)
def summarize_group(g):
    # sample-based stats over incident-window rows
    sample_count = int(g['value'].count())
    sample_mean = g['value'].mean()
    sample_p95 = g['value'].quantile(0.95)
    sample_p5 = g['value'].quantile(0.05)
    # anomalies using threshold-based flags
    mask = g['anomaly_any']
    total_anoms = int(mask.sum())
    if total_anoms == 0:
        earliest = pd.NaT
    else:
        earliest = g.loc[mask, '_ts_dt'].min()
    saw_high = bool(g['anomaly_high'].any())
    saw_low = bool(g['anomaly_low'].any())
    if saw_high and saw_low:
        a_type = 'both'
    elif saw_high:
        a_type = 'high'
    elif saw_low:
        a_type = 'low'
    else:
        a_type = 'none'
    # longest consecutive anomaly streak in minutes
    times = g.loc[mask, '_ts_dt'].sort_values()
    if times.empty:
        longest = 0
    else:
        secs = (times.view('int64') // 10**9).astype(np.int64)
        if len(secs) == 1:
            longest = 1
        else:
            diffs = np.diff(secs)
            breaks = np.where(diffs != 60)[0]
            starts = np.concatenate(([0], breaks + 1))
            ends = np.concatenate((breaks, [len(secs)-1]))
            lengths = ends - starts + 1
            longest = int(lengths.max())
    # include thresholds (attached earlier) - take first if present
    threshold_mean = g['threshold_mean'].iloc[0] if 'threshold_mean' in g.columns else np.nan
    threshold_p95 = g['threshold_p95'].iloc[0] if 'threshold_p95' in g.columns else np.nan
    threshold_p5 = g['threshold_p5'].iloc[0] if 'threshold_p5' in g.columns else np.nan

    return pd.Series({
        'sample_count': sample_count,
        'mean': sample_mean,
        'p95': sample_p95,
        'p5': sample_p5,
        'total_anomalies': total_anoms,
        'earliest_anomaly_timestamp_UTC+8': earliest,
        'anomaly_type': a_type,
        'longest_consecutive_anomaly_streak_minutes': longest,
        'threshold_mean': threshold_mean,
        'threshold_p95': threshold_p95,
        'threshold_p5': threshold_p5
    })

node6_kpi_anomalies = (
    node6_rows
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .apply(lambda g: summarize_group(g))
    .reset_index()
)

# Ensure timestamp column is timezone-aware Asia/Shanghai
def ensure_tz_or_nat(ts):
    if pd.isna(ts):
        return pd.NaT
    ts = pd.to_datetime(ts)
    if ts.tzinfo is None:
        return ts.tz_localize(tz)
    return ts.tz_convert(tz)

node6_kpi_anomalies['earliest_anomaly_timestamp_UTC+8'] = node6_kpi_anomalies['earliest_anomaly_timestamp_UTC+8'].apply(ensure_tz_or_nat)

# 5) Prepare top 20 rows sorted by total_anomalies desc and select requested columns
display_cols = [
    'cmdb_id',
    'kpi_name',
    'sample_count',
    'mean',
    'p95',
    'p5',
    'total_anomalies',
    'earliest_anomaly_timestamp_UTC+8',
    'anomaly_type',
    'longest_consecutive_anomaly_streak_minutes'
]

top20_node6 = (
    node6_kpi_anomalies
    .sort_values('total_anomalies', ascending=False)
    .loc[:, display_cols]
    .head(20)
    .reset_index(drop=True)
)

# Round numeric columns for compactness
for c in ['mean', 'p95', 'p5']:
    if c in top20_node6.columns:
        top20_node6[c] = top20_node6[c].round(6)

# Export variables to kernel: node6_kpi_anomalies (full) and show top20
node6_kpi_anomalies, top20_node6

```
Out[14]:
```
Summary of node-6 KPI anomaly analysis (plain English):

- I stored the per-(cmdb_id,kpi_name) results in variable node6_kpi_anomalies after joining node-6 thresholds with incident-window rows and flagging per-row anomalies.

- Scope: 59 distinct (node-6, kpi_name) pairs were analyzed over the 2022-03-21 12:00–12:30 window.

- Persistence: many KPIs (a large subset) had total_anomalies = 31, i.e., anomalous at every minute of the 31‑minute incident window. Earliest anomalies for these pairs begin at 2022-03-21 12:00:00+08.

- Affected resource areas (examples from the top results):
  - Disk / filesystem / I/O: system.disk.readonly, system.disk.total, system.io.r_s, system.io.rkb_s, system.fs.inodes.in_use, etc. — many show full-window anomalies.
  - Memory / swap: system.mem.total, system.swap.* metrics — persistent anomalies present.
  - Network: system.tcp.retrans_pct, system.udp.connect.num, net packet error counters — anomalous across the window.
  - CPU: system.cpu.pct_usage, system.cpu.user, system.cpu.system, system.cpu.iowait — show anomalies (some with shorter streaks than full window).
  - Processes: system.process.zombie.num is elevated (anomaly_type = "high") and persistent, which is notable.

- Interpretation: Node-6 exhibits broad, persistent anomalies spanning I/O/disk, memory/swap, network, CPU, and process counts. The pattern (many I/O and disk-related KPIs anomalous for the entire window) points to a node-level resource / I/O problem rather than a transient app-only issue.

- Recommended next checks: node-6 disk I/O and disk space, disk queue lengths, underlying storage health/latency, and process states (zombies) on node-6.

The original code execution output of IPython Kernel is also provided below for reference:

(    index cmdb_id                kpi_name  sample_count       mean     p95     p5  total_anomalies earliest_anomaly_timestamp_UTC+8 anomaly_type  longest_consecutive_anomaly_streak_minutes  threshold_mean  threshold_p95  threshold_p5
0       0  node-6        ping.can_connect            31   1.000000   1.000   1.00               31        2022-03-21 12:00:00+08:00         both                                          31        1.000000           1.00          1.00
1       1  node-6       system.cpu.iowait            31   0.040645   0.180   0.00               13        2022-03-21 12:00:00+08:00         both                                           3        0.042295           0.15          0.00
2       2  node-6    system.cpu.pct_usage            31   9.932258  15.840   7.72                8        2022-03-21 12:00:00+08:00         both                                           4        9.711311          14.86          7.80
3       3  node-6       system.cpu.system            31   4.518387   4.985   3.96                6        2022-03-21 12:03:00+08:00         both                                           3        4.572951           5.15          4.04
4       4  node-6         system.cpu.user            31   5.371613  11.025   3.74                8        2022-03-21 12:00:00+08:00         both                                           4        5.092623           8.53          3.74
..    ...     ...                     ...           ...        ...     ...    ...              ...                              ...          ...                                         ...             ...            ...           ...
54     54  node-6       system.swap.total            31   0.000000   0.000   0.00               31        2022-03-21 12:00:00+08:00         both                                          31        0.000000           0.00          0.00
55     55  node-6        system.swap.used            31   0.000000   0.000   0.00               31        2022-03-21 12:00:00+08:00         both                                          31        0.000000           0.00          0.00
56     56  node-6    system.swap.used_pct            31   0.000000   0.000   0.00               31        2022-03-21 12:00:00+08:00         both                                          31        0.000000           0.00          0.00
57     57  node-6  system.tcp.retrans_pct            31   0.010000   0.010   0.01               31        2022-03-21 12:00:00+08:00         both                                          31        0.010000           0.01          0.01
58     58  node-6  system.udp.connect.num            31  11.032258  11.000  11.00               31        2022-03-21 12:00:00+08:00         both                                          31       11.016393          11.00         11.00

[59 rows x 14 columns],    cmdb_id                       kpi_name  sample_count          mean           p95            p5  total_anomalies earliest_anomaly_timestamp_UTC+8 anomaly_type  longest_consecutive_anomaly_streak_minutes
0   node-6               ping.can_connect            31  1.000000e+00  1.000000e+00  1.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
1   node-6           system.disk.readonly            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
2   node-6              system.disk.total            31  5.514861e+09  8.303561e+09  5.101720e+09               31        2022-03-21 12:00:00+08:00         both                                          31
3   node-6              system.io.r_await            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
4   node-6                  system.io.r_s            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
5   node-6                system.io.rkb_s            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
6   node-6        system.fs.inodes.in_use            31  1.149355e+00  1.170000e+00  1.010000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
7   node-6         system.udp.connect.num            31  1.103226e+01  1.100000e+01  1.100000e+01               31        2022-03-21 12:00:00+08:00         both                                          31
8   node-6   system.net.packets_out.error            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
9   node-6    system.net.packets_in.error            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
10  node-6               system.mem.total            31  3.211700e+04  3.211700e+04  3.211700e+04               31        2022-03-21 12:00:00+08:00         both                                          31
11  node-6       system.net.udp.in_errors            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
12  node-6  system.net.udp.rcv_buf_errors            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
13  node-6  system.net.udp.snd_buf_errors            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
14  node-6           system.os.nofile.max            31  6.553600e+06  6.553600e+06  6.553600e+06               31        2022-03-21 12:00:00+08:00         both                                          31
15  node-6      system.process.zombie.num            31  3.250000e+02  3.250000e+02  3.250000e+02               31        2022-03-21 12:00:00+08:00         high                                          31
16  node-6                 system.swap.so            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
17  node-6                 system.swap.si            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
18  node-6               system.swap.free            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31
19  node-6               system.swap.used            31  0.000000e+00  0.000000e+00  0.000000e+00               31        2022-03-21 12:00:00+08:00         both                                          31)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```